In [ ]:
import torch


def debug_print(name, tensor):
    """打印张量的全套体检报告"""
    print(f"{name}: shape={tensor.shape}, dtype={tensor.dtype}, "
          f"device={tensor.device}, "
          f"min={tensor.min().item():.4f}, max={tensor.max().item():.4f}, "
          f"mean={tensor.mean().item():.4f}, "
          f"has_nan={tensor.isnan().any().item()}")

# 现场测一个假张量（假装是一批训练数据）
x = torch.randn(4, 3, 32, 32)          # 4张32x32的3通道图
debug_print("输入批次 x", x)

y = x.cuda()          # 把张量搬上显卡（你的 RTX 5070）
debug_print("搬到GPU后 y", y)    # 注意 device= 变成了什么

In [ ]:
print(f"已显式分配: {torch.cuda.memory_allocated() / 1e9:.3f} GB")
print(f"已缓存预留: {torch.cuda.memory_reserved() / 1e9:.3f} GB")
print(torch.cuda.get_device_name(0))
print(torch.cuda.device_count())                  # CUDA 能看到几张卡？预期：1
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")   # 显存：预期 ~8.2 GB

## Part3 logging

In [2]:
# ========== Part3 logging 完整实验（可直接运行） ==========

# ① 导入标准库 logging —— 注意：自带的，不用 uv pip install
import logging

# ② 补上课程片段里"没定义"的变量（真实训练中它们来自优化器/dataloader）
lr = 0.001          # 学习率（float）
batch_size = 32     # 批大小（int）
step = 100          # 当前训练步数（int）
loss = 2.3026       # 当前损失（float，这里先模拟一个正常数值）

# ③ 配置 logging：级别=INFO、格式=时间戳+级别+消息、双 handler（文件+屏幕）
logging.basicConfig(
    level=logging.INFO,                                 # 门槛：INFO 及以上放行
    format="%(asctime)s [%(levelname)s] %(message)s",   # 播音稿模板
    handlers=[                                          # 出口清单（双出口）
        logging.FileHandler("training.log"),            # 喇叭1：写进文件 training.log
        logging.StreamHandler()                         # 喇叭2：写进屏幕
    ],
    force=True                                          # ★老师加的防坑参数（见下）
)
logger = logging.getLogger(__name__)                    # 领一个话筒

# ④ 三种级别喊话（变量已定义，直接能跑）
logger.info("Starting training: lr=%.4f, batch_size=%d", lr, batch_size)
logger.warning("Loss spike detected: %.4f at step %d", loss, step)
logger.error("NaN loss at step %d, stopping", step)

2026-08-31 09:18:54,023 [INFO] Starting training: lr=0.0010, batch_size=32
2026-08-31 09:18:54,024 [WARNING] Loss spike detected: 2.3026 at step 100
2026-08-31 09:18:54,025 [ERROR] NaN loss at step 100, stopping


In [3]:
# 模拟 20 步训练：损失正常下降，但第 7、15 步出现"尖峰"，最后一步 NaN
for step in range(20):
    loss = 2.5 - step * 0.1                 # 假装损失在正常下降
    if step in (7, 15):
        logger.warning("Loss spike detected: %.4f at step %d", loss + 15.0, step)
    else:
        logger.info("Step %d: loss=%.4f", step, loss)

logger.error("NaN loss at step 21, stopping")

2026-08-31 09:29:13,915 [INFO] Step 0: loss=2.5000
2026-08-31 09:29:13,917 [INFO] Step 1: loss=2.4000
2026-08-31 09:29:13,918 [INFO] Step 2: loss=2.3000
2026-08-31 09:29:13,918 [INFO] Step 3: loss=2.2000
2026-08-31 09:29:13,919 [INFO] Step 4: loss=2.1000
2026-08-31 09:29:13,919 [INFO] Step 5: loss=2.0000
2026-08-31 09:29:13,920 [INFO] Step 6: loss=1.9000
2026-08-31 09:29:13,921 [WARNING] Loss spike detected: 16.8000 at step 7
2026-08-31 09:29:13,922 [INFO] Step 8: loss=1.7000
2026-08-31 09:29:13,922 [INFO] Step 9: loss=1.6000
2026-08-31 09:29:13,923 [INFO] Step 10: loss=1.5000
2026-08-31 09:29:13,923 [INFO] Step 11: loss=1.4000
2026-08-31 09:29:13,924 [INFO] Step 12: loss=1.3000
2026-08-31 09:29:13,925 [INFO] Step 13: loss=1.2000
2026-08-31 09:29:13,925 [INFO] Step 14: loss=1.1000
2026-08-31 09:29:13,927 [WARNING] Loss spike detected: 16.0000 at step 15
2026-08-31 09:29:13,928 [INFO] Step 16: loss=0.9000
2026-08-31 09:29:13,929 [INFO] Step 17: loss=0.8000
2026-08-31 09:29:13,929 [INFO]